In [2]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 23.5 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import string
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torchao.quantization import quantize_, Int4WeightOnlyConfig
from torchao.quantization.qat import QATConfig

In [2]:
DATASET_PATH = './COGNITIVE DISTORTION DATASET IN BAHASA INDONESIA COMPLETE.csv'

df = pd.read_csv(DATASET_PATH)
df.head()

,TEXT,DATA STATUS,FIRST ANNOTATOR,SECOND ANNOTATOR
0,ketika saya sudah berusaha semaksimal mungkin ...,RAW-ORI,All-or-nothing,All-or-nothing
1,Saya ketika SD hingga SMP dulu beranggapan bah...,RAW-ORI,All-or-nothing,All-or-nothing
2,Saya berfikir jika tidak lolos MBKM maka masa ...,RAW-ORI,All-or-nothing,All-or-nothing
3,"Di desa saya dulu ada dua kelompok pertemanan,...",RAW-ORI,All-or-nothing,All-or-nothing
4,Saya hanya berteman dengan orang yang tidak me...,RAW-ORI,All-or-nothing,All-or-nothing


In [3]:
df['FIRST ANNOTATOR'].value_counts()

,count
FIRST ANNOTATOR,
No Distortion,2246
Labeling,390
Mind Reading,385
Should statement,371
Personalization and Blame,284
Discounting the positives,241
Overgeneralization,215
All-or-nothing,200
Mental filter,200


In [4]:
df['SECOND ANNOTATOR'].value_counts()

,count
SECOND ANNOTATOR,
No Distortion,2246
Labeling,390
Mind Reading,385
Should statement,371
Personalization and Blame,284
Discounting the positives,241
Overgeneralization,215
All-or-nothing,200
Mental filter,200


In [5]:
enc = LabelEncoder()

num_labels = enc.fit_transform(df['FIRST ANNOTATOR'])
df['LABEL'] = num_labels

In [6]:
df.head()

,TEXT,DATA STATUS,FIRST ANNOTATOR,SECOND ANNOTATOR,LABEL
0,ketika saya sudah berusaha semaksimal mungkin ...,RAW-ORI,All-or-nothing,All-or-nothing,0
1,Saya ketika SD hingga SMP dulu beranggapan bah...,RAW-ORI,All-or-nothing,All-or-nothing,0
2,Saya berfikir jika tidak lolos MBKM maka masa ...,RAW-ORI,All-or-nothing,All-or-nothing,0
3,"Di desa saya dulu ada dua kelompok pertemanan,...",RAW-ORI,All-or-nothing,All-or-nothing,0
4,Saya hanya berteman dengan orang yang tidak me...,RAW-ORI,All-or-nothing,All-or-nothing,0


In [7]:
num_labels = len(enc.classes_)
label2id = dict(zip(enc.classes_, range(num_labels)))
id2label = dict(enumerate(enc.classes_))

print("Label Mapping:", label2id)

Label Mapping: {'All-or-nothing': 0, 'Discounting the positives': 1, 'Emotional Reasoning': 2, 'Fortune-telling': 3, 'Labeling': 4, 'Magnification or Minimization': 5, 'Mental filter': 6, 'Mind Reading': 7, 'No Distortion': 8, 'Overgeneralization': 9, 'Personalization and Blame': 10, 'Should statement': 11}


In [8]:
from transformers import AutoTokenizer, AutoModel, TrainingArguments, AutoModelForSequenceClassification
from transformers import Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [9]:
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p1")

config.json:   0%|          | 0.00/1.53k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/229k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [10]:
X = df['TEXT'].apply(lambda x: x.replace('$',''))
y = df['LABEL']

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=42, test_size=0.1)

In [11]:
def tokenize(batch):
    return tokenizer(batch['TEXT'], padding=True, truncation=True, max_length=128)


In [12]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = logits.argmax(axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted"),
        "precision": precision_score(labels, predictions, average="weighted"),
        "recall": recall_score(labels, predictions, average="weighted")
    }

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    "indobenchmark/indobert-base-p1",
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

base_config = Int4WeightOnlyConfig(group_size=32)
quantize_(model, QATConfig(base_config, step="prepare"))

[transformers] You passed `num_labels=12` which is incompatible to the `id2label` map of length `5`.


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  498MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  498MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
X = df['TEXT'].apply(lambda x: x.replace('$',''))
y = df['LABEL']

x_train, x_test, y_train, y_test = train_test_split(X,y, random_state=42, test_size=0.1)

In [17]:
from datasets import Dataset, DatasetDict

train_df = pd.DataFrame({
    'TEXT':x_train, 'LABEL':y_train
})
val_df = pd.DataFrame({
    'TEXT':x_test, 'LABEL':y_test
})

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

train_dataset = train_dataset.rename_column("LABEL", "labels")
val_dataset = val_dataset.rename_column("LABEL", "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

dataset_dict = DatasetDict({
    'train': train_dataset,
    'val': val_dataset
})

training_args = TrainingArguments(
    output_dir=f"./result",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.05,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_accuracy",
    greater_is_better=True,
    remove_unused_columns=False
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_dict["train"],
    eval_dataset=dataset_dict["val"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
quantize_(model, QATConfig(base_config, step="convert"))

val_metrics = trainer.evaluate(eval_dataset=dataset_dict["val"])
train_metrics = trainer.evaluate(eval_dataset=dataset_dict["train"])


Map:   0%|          | 0/4492 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,1.400714,1.102329,0.632000,0.624905,0.662572,0.632000
2,0.754404,0.993136,0.670000,0.656816,0.676332,0.670000
3,0.454559,0.979083,0.708000,0.703022,0.706946,0.708000
4,0.246271,1.123277,0.716000,0.712487,0.716294,0.716000
5,0.142613,1.184404,0.712000,0.705812,0.710622,0.712000


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

ImportError: Requires mslk >= 1.0.0

In [ ]:
test = pd.DataFrame({
    'x':x_test, 'y':y_test
})

In [ ]:
from transformers import pipeline

def get_model(model_path):
    model = AutoModelForSequenceClassification.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    pipe = pipeline("text-classification", model=model, tokenizer=tokenizer)
    return pipe

In [ ]:
pred_model = get_model('./result/checkpoint-1124')
model_probs = []
outputs = pred_model(list(test['x']), top_k=12)
for output in outputs:
    sorted_output = sorted(output, key=lambda x: x["label"])
    probs = [item["score"] for item in sorted_output]
    model_probs.append(probs)

all_fold_preds.append(model_probs)

final_probs = np.mean(all_fold_preds, axis=0)
final_preds = np.argmax(final_probs, axis=1)
# ensemble_preds = np.mean(all_fold_preds, axis=0)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report, accuracy_score
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

print(f"Ensemble Accuracy: {accuracy_score(test['y'], final_preds):.4f}")
print("\nClassification Report:")
print(classification_report(test['y'], final_preds))
cm = confusion_matrix(test['y'], final_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=list(id2label.keys()))
disp.plot(cmap=plt.cm.Blues)
plt.show()
